#Clinical Trial Dropout Prediction (03_Feature_Engineering_and_Preprocessing)

## 1. Notebook Objective

The objective of this notebook is to prepare the clinical trial dataset for Machine Learning.

Based on the findings from the Exploratory Data Analysis (EDA), this stage will focus on:

- Selecting relevant features for predictive modeling.
- Addressing identified data-quality issues.
- Transforming variables into appropriate data types.
- Creating meaningful derived features.
- Separating features from the target variable.
- Splitting the dataset into training and testing sets.
- Building a reproducible preprocessing pipeline.

The resulting preprocessing workflow will be used in the subsequent Machine Learning modeling stage.


## 2. Decisions Based on EDA

The feature selection and preprocessing decisions are based on the findings from the previous Data Understanding and Exploratory Data Analysis stages.

| Variable | Decision | Justification |
|---|---|---|
| `Subject_ID` |  ❌ Remove | Identifier with no meaningful predictive information. |
| `Site_ID` |  ❌ Remove | High-cardinality variable that may lead to site-specific memorization rather than generalizable patient-level patterns. |
| `Age` | ✅ Keep | Relevant demographic characteristic. |
| `Gender` | ✅ Keep | Relevant demographic characteristic. |
| `Enrollment_Date` |🆕 Transform | The original date is not directly suitable for the model. A temporal feature will be derived from it. |
| `Treatment_Group` | ✅ Keep | Relevant clinical trial characteristic. |
| `Adverse_Events` | ✅ Keep | Clinical variable that may contain predictive information despite its weak individual association with dropout. |
| `Dropout` | 🎯 Target | Binary target variable to be predicted. |
| `Systolic_BP` | ✅ Keep | Relevant clinical measurement. |
| `Diastolic_BP` | ✅ Keep | Relevant clinical measurement. |
| `Cholesterol_Level` | ✅ Keep | Relevant clinical measurement. |
| `Time_Since_Start`  | 🆕 Create    | Represents participant's temporal position within the trial                                               |
                              

### Data Quality Decision

The dataset documentation indicates that several anomalies were intentionally introduced to simulate real-world clinical trial data quality issues.

During the EDA, potentially anomalous age values were identified. Given the clinical context and the nature of the dataset, these observations will be treated as missing values (`NaN`) rather than removing the corresponding participants.

The missing values will be handled during preprocessing using median imputation.

### Feature Engineering Decision

`Enrollment_Date` will be transformed into a numerical feature called `Time_Since_Start`, representing the number of days between participant enrollment and the beginning of the clinical trial.

The original `Enrollment_Date` variable will subsequently be removed from the model features.

`Site_ID` will be excluded from the predictive features because it is a high-cardinality identifier that may encourage site-specific memorization rather than generalizable patient-level patterns.


###Conect to Drive

In [2]:
#Conect to drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.listdir("/content/drive/MyDrive/Bioinformatica/Data Science/Clinical_trial_dropout_prediction/data/raw")

Mounted at /content/drive


['synthetic_clinical_trial_data.csv']

##3. Import libraries and Load Dataset

In [3]:
# Import libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

#Load Dataset
df = pd.read_csv("/content/drive/MyDrive/Bioinformatica/Data Science/Clinical_trial_dropout_prediction/data/raw/synthetic_clinical_trial_data.csv")

In [9]:
#Verify that the dataset was loaded correctly
df.head()



,Subject_ID,Site_ID,Age,Gender,Enrollment_Date,Treatment_Group,Adverse_Events,Dropout,Systolic_BP,Diastolic_BP,Cholesterol_Level
0,1,49,54,Male,1/1/22,Drug A,0,0,117,74,229
1,2,37,44,Male,1/2/22,Placebo,1,0,111,57,173
2,3,1,58,Male,1/3/22,Drug A,0,1,122,89,220
3,4,25,48,Male,1/4/22,Drug B,0,0,122,85,175
4,5,10,57,Female,1/5/22,Drug A,2,0,105,90,185


## 4. Data Type Conversion

The `Enrollment_Date` variable was initially stored as an object data type. Since it represents a calendar date, it will be converted to the `datetime` format to allow temporal feature engineering.

In [10]:
# ==========================
# Enrollment_Date: Data Type Conversion
# ==========================

df["Enrollment_Date"] = pd.to_datetime(
    df["Enrollment_Date"]
)

df["Enrollment_Date"].dtype

/tmp/ipykernel_1775/2418017204.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Enrollment_Date"] = pd.to_datetime(


dtype('<M8[ns]')

## 5. Data Quality Treatment

During the EDA, several unusually low age values were identified. The dataset documentation indicates that synthetic anomalies were intentionally included to simulate real-world clinical trial data quality issues.

Given the clinical context of a Phase III hypertension trial, ages below 18 years are considered potential data-quality anomalies.

Rather than removing the affected participants, only the `Age` values will be replaced with missing values (`NaN`). This preserves the remaining information associated with these participants.

The missing age values will be handled later through the preprocessing pipeline using median imputation, with the median learned exclusively from the training data.

In [19]:
# ==========================
# Age: Data Quality Assessment
# ==========================

df[df["Age"] < 18]["Age"].value_counts().sort_index()

,count
Age,
10.0,1
11.0,1
12.0,1
13.0,1
14.0,5
15.0,1
17.0,1


In [20]:
# ==========================
# Age: Missing Values Check
# ==========================

df["Age"].isna().sum()

np.int64(1)

In [21]:
# ==========================
# Age: Replace Anomalous Values
# ==========================

df.loc[df["Age"] < 18, "Age"] = np.nan

In [22]:
# ==========================
# Age: Validation
# ==========================

print("Missing Age values:", df["Age"].isna().sum())
print("Minimum valid Age:", df["Age"].min())

Missing Age values: 12
Minimum valid Age: 19.0


In [23]:
# ==========================
# Age: Dataset Integrity Check
# ==========================

df.shape

(1000, 11)

## 6. Feature Engineering

### Time Since Trial Start

The `Enrollment_Date` variable contains temporal information that may be relevant for predicting participant dropout. However, the original date is not directly suitable as a numerical model feature.

A new variable, `Time_Since_Start`, will therefore be created to represent the number of days elapsed between the beginning of the trial and each participant's enrollment.

The trial start date is defined as January 1, 2022, corresponding to the earliest enrollment date identified during the EDA.

In [24]:
# ==========================
# Enrollment_Date → Time_Since_Start
# ==========================

TRIAL_START_DATE = pd.Timestamp("2022-01-01")

df["Time_Since_Start"] = (
    df["Enrollment_Date"] - TRIAL_START_DATE
).dt.days

In [25]:
# ==========================
# Time_Since_Start: Validation
# ==========================

df[
    ["Enrollment_Date", "Time_Since_Start"]
].head()

,Enrollment_Date,Time_Since_Start
0,2022-01-01,0
1,2022-01-02,1
2,2022-01-03,2
3,2022-01-04,3
4,2022-01-05,4


In [26]:
# ==========================
# Time_Since_Start: Range
# ==========================

df["Time_Since_Start"].agg(["min", "max"])

,Time_Since_Start
min,0
max,999


In [29]:
# ==========================
# Remove Original Date Variable
# ==========================

df = df.drop(columns=["Enrollment_Date"])



KeyError: "['Enrollment_Date'] not found in axis"

In [30]:
df.columns

Index(['Subject_ID', 'Site_ID', 'Age', 'Gender', 'Treatment_Group',
       'Adverse_Events', 'Dropout', 'Systolic_BP', 'Diastolic_BP',
       'Cholesterol_Level', 'Time_Since_Start'],
      dtype='object')

## 7. Feature and Target Separation

The target variable for this project is `Dropout`, which indicates whether a participant dropped out of the clinical trial.

`Subject_ID` and `Site_ID` will be excluded from the predictive features:

- `Subject_ID` is an individual identifier and does not provide meaningful predictive information.
- `Site_ID` is excluded because its high cardinality may encourage the model to learn site-specific patterns rather than generalizable patient-level relationships.

The remaining variables will be used as candidate predictive features.

In [31]:
# ==========================
# Feature / Target Separation
# ==========================

# Remove identifiers excluded from modeling
df_model = df.drop(columns=["Subject_ID", "Site_ID"])

# Separate features and target
X = df_model.drop(columns=["Dropout"])
y = df_model["Dropout"]

In [32]:
# ==========================
# Feature / Target Validation
# ==========================

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 8)
y shape: (1000,)


In [33]:
X.columns

Index(['Age', 'Gender', 'Treatment_Group', 'Adverse_Events', 'Systolic_BP',
       'Diastolic_BP', 'Cholesterol_Level', 'Time_Since_Start'],
      dtype='object')

In [34]:
y.name

'Dropout'

In [35]:
X.isna().sum()

,0
Age,12
Gender,0
Treatment_Group,0
Adverse_Events,0
Systolic_BP,0
Diastolic_BP,0
Cholesterol_Level,0
Time_Since_Start,0


## 8. Train / Test Split

The dataset will be divided into training and testing sets before applying preprocessing transformations.

The training set will be used to fit the preprocessing pipeline and train the Machine Learning models, while the testing set will be kept completely independent for final model evaluation.

Because the target variable is imbalanced (`Dropout = 0`: 83.9%, `Dropout = 1`: 16.1%), stratified sampling will be used to preserve a similar class distribution in both subsets.

In [36]:
# ==========================
# Train / Test Split
# ==========================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [37]:
# ==========================
# Split Validation
# ==========================

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800, 8)
X_test: (200, 8)
y_train: (800,)
y_test: (200,)


In [38]:
# ==========================
# Target Distribution
# ==========================

print("Overall:")
print(y.value_counts(normalize=True))

print("\nTraining set:")
print(y_train.value_counts(normalize=True))

print("\nTesting set:")
print(y_test.value_counts(normalize=True))

Overall:
Dropout
0    0.839
1    0.161
Name: proportion, dtype: float64

Training set:
Dropout
0    0.83875
1    0.16125
Name: proportion, dtype: float64

Testing set:
Dropout
0    0.84
1    0.16
Name: proportion, dtype: float64


## 9. Preprocessing Strategy

The features are divided into numerical and categorical variables.

Numerical variables will be imputed using the median and standardized using `StandardScaler`.

Categorical variables will be imputed using the most frequent category and transformed using one-hot encoding.

All preprocessing operations will be incorporated into a `ColumnTransformer` and later combined with the Machine Learning models through Scikit-learn pipelines.

In [40]:
# ==========================
# Feature Types
# ==========================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "Age",
    "Adverse_Events",
    "Systolic_BP",
    "Diastolic_BP",
    "Cholesterol_Level",
    "Time_Since_Start"
]

categorical_features = [
    "Gender",
    "Treatment_Group"
]


##9.1 Numerical preprocessing
### Numerical Features

Numerical variables will undergo two preprocessing steps:

1. Missing values will be imputed using the median.
2. Variables will be standardized using `StandardScaler`.

Median imputation is used because it is less sensitive to extreme values than mean imputation and is appropriate for the clinical variables in this dataset.

Standardization transforms the numerical variables to a comparable scale, which is particularly important for distance-based and regularized Machine Learning algorithms.

In [41]:
# ==========================
# Numerical Preprocessing
# ==========================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

##9.2 Categorical preprocessing
### Categorical Features

Categorical variables will first be checked for missing values and then transformed using one-hot encoding.

Missing categorical values will be replaced with the most frequent category observed in the training data.

One-hot encoding converts categorical variables into numerical binary features that can be used by Machine Learning algorithms.

In [42]:
# ==========================
# Categorical Preprocessing
# ==========================

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

##9.3 Combine both transformers
### ColumnTransformer

The numerical and categorical preprocessing pipelines are combined using `ColumnTransformer`.

This allows each group of variables to receive the appropriate preprocessing while maintaining a single reproducible transformation workflow.

In [43]:
# ==========================
# ColumnTransformer
# ==========================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [44]:
# ==========================
# Preprocessor Validation
# ==========================

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Adverse_Events', 'Systolic_BP',
                                  'Diastolic_BP', 'Cholesterol_Level',
                                  'Time_Since_Start']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'Treatment_Group'])])

## 10. Save Preprocessing Objects

The train/test split and preprocessing configuration created in this notebook will be reused in the Machine Learning modeling stage.

To maintain a reproducible workflow and avoid repeating the previous transformations, the following objects will be saved:

- `X_train`: training features.
- `X_test`: testing features.
- `y_train`: training target.
- `y_test`: testing target.
- `preprocessor`: the preprocessing configuration defined using `ColumnTransformer`.

The preprocessor will be saved before fitting. It will be fitted together with each Machine Learning model in the next notebook, using only the training data.

This approach prevents data leakage and ensures that the preprocessing applied to the test set is based exclusively on information learned from the training set.



##10.1 Import joblib and save objects

In [45]:
# ==========================
# Save Preprocessing Objects
# ==========================

import joblib

# ==========================
# Package Preprocessing Objects
# ==========================

preprocessing_objects = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "preprocessor": preprocessor
}

##10.2 Save in Google Drive


In [46]:
# ==========================
# Output Path
# ==========================

OUTPUT_PATH = "/content/drive/MyDrive/Bioinformatica/Data Science/Clinical_trial_dropout_prediction/data/processed/preprocessing_objects.pkl"

In [47]:
# ==========================
# Save Objects
# ==========================

joblib.dump(
    preprocessing_objects,
    OUTPUT_PATH
)

['/content/drive/MyDrive/Bioinformatica/Data Science/Clinical_trial_dropout_prediction/data/processed/preprocessing_objects.pkl']

In [48]:
# ==========================
# Save Validation
# ==========================

import os

os.path.exists(OUTPUT_PATH)

True

In [49]:
# ==========================
# Load Validation
# ==========================

test_objects = joblib.load(OUTPUT_PATH)

test_objects.keys()

dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'preprocessor'])

## 11. Modeling Strategy

Two classification algorithms will be evaluated in the next stage of the project.

### Logistic Regression — Baseline

Logistic Regression was selected as the baseline model because it is a simple, interpretable algorithm for binary classification. Its coefficients can provide insight into the direction and magnitude of the association between the predictors and dropout probability.

### Random Forest — Non-linear Model

Random Forest was selected as a more flexible model capable of capturing non-linear relationships and interactions between predictors that may not be captured by Logistic Regression.

Comparing both models will allow us to evaluate whether a more complex algorithm provides meaningful improvements over the interpretable baseline.